# Description of Notebook
- This notebook covers code related to Random Forest and XGBoost Model creation as well as evaluation and analysis.

## Model Creation
- Input:
    - processed training data 
    - Target feature: RUL of each unit(remaining cycles; max_cycle - current_cycle = remaining_cycles)
- Output: 
    - Random Forest Model
    - XGBoost Model

## Model Evaluation
- Input:
    - Model
    - Test data
- Output:
    - RMSE for each model
    - MAE for each model
    - Forecast error over time visualization for each model
    - Residual Analysis for each model
    - Feature Importance Analysis (if applicable for models)
    - Comparison of at least 2 forecasting horizons

## Data Preprocessing Here

In [1]:
from collections.abc import Hashable, Mapping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
# PreProcessing and Feature Engineering Functions
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """Placeholder for future preprocessing pipeline steps."""
    return df.copy()

def _coefficient_of_variation(series: pd.Series) -> float:
    """Return abs(std / mean), or infinity when the mean is effectively zero."""
    mean = series.mean()
    if pd.isna(mean) or np.isclose(mean, 0.0):
        return np.inf

    std = series.std()
    if pd.isna(std):
        return np.inf

    return float(abs(std / mean))

def drop_low_cv_sensors(
    data_dict: Mapping[Hashable, pd.DataFrame],
    threshold: float = 0.1,
) -> tuple[dict[Hashable, pd.DataFrame], list[str]]:
    """Drop sensors whose coefficient of variation is below the threshold across all datasets.

    Args:
        data_dict: Mapping of dataset ids to pandas DataFrames. Sensor columns are
            expected to follow the "Sensor <n>" naming convention, while all other
            columns such as unit ids, cycle counts, settings, RUL, and dataset labels
            are preserved.
        threshold: Minimum coefficient of variation required to keep a sensor.

    Returns:
        A tuple containing:
        - a copied dictionary of DataFrames with globally low-CV sensor columns removed
        - a sorted list of sensor column names that were dropped

    A sensor is removed only if its coefficient of variation, computed as abs(std / mean),
    is below ``threshold`` in every DataFrame where that sensor exists.
    """
    filtered_data = {key: df.copy() for key, df in data_dict.items()}

    sensor_cols = sorted(
        {
            col
            for df in filtered_data.values()
            for col in df.columns
            if col.startswith("Sensor")
        },
        key=lambda name: int(name.split()[1]),
    )

    sensors_to_drop: list[str] = []

    for sensor in sensor_cols:
        cv_values = [
            _coefficient_of_variation(df[sensor])
            for df in filtered_data.values()
            if sensor in df.columns
        ]

        if cv_values and all(cv < threshold for cv in cv_values):
            sensors_to_drop.append(sensor)

    for df in filtered_data.values():
        present_sensors = [sensor for sensor in sensors_to_drop if sensor in df.columns]
        if present_sensors:
            df.drop(columns=present_sensors, inplace=True)

    return filtered_data, sensors_to_drop

def compute_RUL(
    data_dict: Mapping[Hashable, pd.DataFrame]
) -> dict[Hashable, pd.DataFrame]:
    """Compute the RUL for each row in the datasets and add it as a new column.
        Args:
        data_dict: Mapping of dataset ids to pandas DataFrames.

    Returns:
        - a copied dictionary of DataFrames with RUL column
     RUL column is computed for each time step of each unit.
    """
    for i in data_dict:
        df = data_dict[i].copy()
        df["RUL"] = df.groupby("Unit Number")["Time, In Cycles"].transform(lambda x: x.max() - x)
        df["Dataset"] = f"FD00{i}" # adding a column to identify which dataset each row belongs to for combined analysis
        data_dict[i] = df
    return data_dict

def compute_lags(
        data_dict: Mapping[Hashable, pd.DataFrame],
        sensor_cols: list[str],
        lags: list[int]
) -> dict[Hashable, pd.DataFrame]:
        """Compute lag features for specified sensor columns and lags, and add them as new columns.
            Args:
            data_dict: Mapping of dataset ids to pandas DataFrames.
            sensor_cols: List of sensor column names to compute lags for.
            lags: List of integer lag values to compute.

        Returns:
            A dictionary of DataFrames with the computed lag features added as new columns.
        Lag features are computed for each time step of each unit, and lagged values are aligned with the current time step."""

        lagged_data = {key: df.copy() for key, df in data_dict.items()}
        for i in data_dict:
            df = data_dict[i].copy()
            # creating lag features for each unit to avoid data leaks across units
            for lag in lags:
                df_lag = df.groupby("Unit Number")[sensor_cols].shift(lag)
                df_lag.columns = [f"{col}_lag{lag}" for col in sensor_cols]

                df = df.join(df_lag)
            lagged_data[i] = df.dropna()
        return lagged_data

def compute_window_features(
    data_dict: Mapping[Hashable, pd.DataFrame],
    sensor_cols: list[str],
    window_size: int
) -> dict[Hashable, pd.DataFrame]:
    """Compute strictly historical rolling window features for each sensor.
        Args:
        data_dict: Mapping of dataset ids to pandas DataFrames.
        sensor_cols: List of sensor column names to compute window features for.
        window_size: Integer size of the rolling window.
    Returns:
    A dictionary of DataFrames with the computed window features added as new columns.
    Window features are computed for each time step of each unit, and rolling calculations are performed separately for each unit to avoid data leaks across units.
     Each row only uses prior time steps; the current time step is excluded from its own window.
     The new columns are named in the format "<sensor>_window{window_size}_mean" and "<sensor>_window{window_size}_std" for the mean and standard deviation features, respectively.
    """
    windowed_data = {key: df.copy() for key, df in data_dict.items()}
    for i in data_dict:
        df = data_dict[i].copy()
        # creating rolling window features for each unit to avoid data leaks across units
        for sensor in sensor_cols:
            history = df.groupby("Unit Number")[sensor].shift(1)
            df[f"{sensor}_window{window_size}_mean"] = history.groupby(df["Unit Number"]).transform(
                lambda x: x.rolling(window=window_size, min_periods=1).mean()
            )
            df[f"{sensor}_window{window_size}_std"] = history.groupby(df["Unit Number"]).transform(
                lambda x: x.rolling(window=window_size, min_periods=1).std()
            )
        windowed_data[i] = df.dropna()
    return windowed_data

def parse_data() -> dict[int, pd.DataFrame]:
    """Parse the original CMAPSS datasets from the text files and return a dictionary of DataFrames.
    """
    data_dict = {}
    for i in range(1,5):
        df = pd.read_csv(f"CMAPSSData/train_FD00{str(i)}.txt", sep=" ", header=None)
        df = df.drop(columns=[26, 27])  # Remove the last two empty columns
        df.columns = ["Unit Number", "Time, In Cycles", "Setting 1", "Setting 2", "Setting 3"] + [f"Sensor {i}" for i in range(1, 22)]
        data_dict[i] = df
    return data_dict

def clip_RUL(data_dict: Mapping[Hashable, pd.DataFrame], max_RUL: int = 125) -> dict[Hashable, pd.DataFrame]:
    """Clip the RUL values in the datasets to a maximum value.

    Args:
        data_dict: Mapping of dataset ids to pandas DataFrames, each containing an "RUL" column.
        max_RUL: Maximum RUL value to clip to. Any RUL values above this will be set to max_RUL.

    Returns:
        A dictionary of DataFrames with the RUL values clipped to the specified maximum.
    """
    clipped_data = {key: df.copy() for key, df in data_dict.items()}
    for key, df in clipped_data.items():
        if "RUL" in df.columns:
            df["RUL"] = df["RUL"].clip(upper=max_RUL)
    return clipped_data

def train_val_split(data_dict: Mapping[Hashable, pd.DataFrame], test_size: float = 0.3) -> dict[Hashable, dict[Hashable, pd.DataFrame]]:
    """Split the datasets into training and validation sets."""
    train_dict = {}
    val_dict = {}
    
    for key, df in data_dict.items():
        units = df["Unit Number"].unique()
        train_units, val_units = train_test_split(units, test_size=test_size, random_state=42)
        
        train_dict[key] = df[df["Unit Number"].isin(train_units)].copy()
        val_dict[key] = df[df["Unit Number"].isin(val_units)].copy()
        
    split_dict = {"train": train_dict, "val": val_dict}
    return split_dict

def standardize_data(split_dict: Mapping[Hashable, dict[Hashable, pd.DataFrame]]) -> dict[Hashable, pd.DataFrame]:
    """Prepares the data for LSTM by creating sequences.

    Returns separated and scaled features and targets, along with their respective scalers.
    """
    train_df = pd.concat(split_dict["train"].values())
    val_df = pd.concat(split_dict["val"].values())
    
    drop_cols = ["Unit Number", "Dataset", "RUL"]
    
    # Separate features (X) and target (y)
    X_train = train_df.drop(columns=drop_cols, errors="ignore")
    y_train = train_df[["RUL"]]
    
    X_val = val_df.drop(columns=drop_cols, errors="ignore")
    y_val = val_df[["RUL"]]

    feature_scaler = StandardScaler()
    X_train_scaled = feature_scaler.fit_transform(X_train)
    X_val_scaled = feature_scaler.transform(X_val)
    
    target_scaler = StandardScaler()
    y_train_scaled = target_scaler.fit_transform(y_train)
    y_val_scaled = target_scaler.transform(y_val)

    out_dict = {}
    out_dict["X_train_scaled"] = X_train_scaled
    out_dict["y_train_scaled"] = y_train_scaled
    out_dict["X_val_scaled"] = X_val_scaled
    out_dict["y_val_scaled"] = y_val_scaled
    out_dict["feature_scaler"] = feature_scaler
    out_dict["target_scaler"] = target_scaler
    return out_dict

def pipeline_A(data_dict: Mapping[Hashable, pd.DataFrame]) -> dict[Hashable, dict[Hashable, pd.DataFrame]]:
    """Example pipeline that applies the preprocessing steps in sequence."""
    processed_data, dropped_sensors = drop_low_cv_sensors(data_dict, threshold=0.05)
    processed_data = compute_RUL(processed_data)
    sensor_cols = [col for col in data_dict[1].columns.drop(dropped_sensors) if col.startswith("Sensor")]
    processed_data = compute_lags(processed_data, sensor_cols=sensor_cols, lags=[1, 2, 3])
    processed_data = compute_window_features(processed_data, sensor_cols=sensor_cols, window_size=5)
    processed_data = clip_RUL(processed_data, max_RUL=125)
    split_dict = train_val_split(processed_data, test_size=0.3)
    return split_dict

def roll_mean_smooth(data_dict: Mapping[Hashable, pd.DataFrame], window_size: int = 5) -> dict[Hashable, pd.DataFrame]:
    for i in data_dict:
        df = data_dict[i].copy()
        sensor_cols = [col for col in df.columns if col.startswith("Sensor")]
        for sensor in sensor_cols:
            df[f"{sensor}_rolling_mean"] = df[sensor].rolling(window=window_size).mean()
        data_dict[i] = df
    return data_dict

def exp_smooth(data_dict: Mapping[Hashable, pd.DataFrame], window_size: int = 5) -> dict[Hashable, pd.DataFrame]:
    for i in data_dict:
        df = data_dict[i].copy()
        sensor_cols = [col for col in df.columns if col.startswith("Sensor")]
        for sensor in sensor_cols:
            df[f"{sensor}_exp_smooth"] = df[sensor].ewm(span=window_size).mean()
        data_dict[i] = df
    return data_dict


In [3]:
# Load and preprocess the data
data_dict = parse_data()
split_results = pipeline_A(data_dict)
scaled_output = standardize_data(split_results)

X_train = scaled_output["X_train_scaled"]
y_train = scaled_output["y_train_scaled"]
X_val   = scaled_output["X_val_scaled"]
y_val   = scaled_output["y_val_scaled"]

f_scaler = scaled_output["feature_scaler"]
t_scaler = scaled_output["target_scaler"]

## Evaluation Functions

In [4]:
import mlflow
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
def forecast_error_overtime_plot(y_true, y_pred):
    plt.figure(figsize=(12, 6))
    plt.scatter(y_true, y_pred, alpha=0.5)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Perfect Prediction')
    plt.title("Predicted vs Actual RUL")
    plt.xlabel("Actual RUL")
    plt.ylabel("Predicted RUL")
    plt.legend()
    plt.grid()
    plt.show()

def residuals_plot(residuals, y_pred):
    plt.figure(figsize=(12, 6))
    plt.scatter(y_pred, residuals, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.title("Residuals vs Predicted Values")
    plt.xlabel("Predicted RUL")
    plt.ylabel("Residuals")
    plt.grid()
    plt.show()

def residuals_histogram(residuals):
    plt.figure(figsize=(12, 6))
    sns.histplot(residuals, bins=30, kde=True)
    plt.title("Distribution of Residuals")
    plt.xlabel("Residuals")
    plt.ylabel("Frequency")
    plt.grid()
    plt.show()

##########################################################################
# for use with final models #
##########################################################################
def residuals_analysis(y_true, y_pred):
    # for use on final models after tuning
    residuals = y_true - y_pred
    residuals_plot(residuals, y_pred)
    residuals_histogram(residuals)

def evaluate_model(model, X_test, y_test):
    # for use on final models after tuning
    predictions = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, predictions)
    mse = mean_absolute_error(y_test, predictions)
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Mean Absolute Error: {mse}") 
    display(model.feature_importances_)
    forecast_error_overtime_plot(y_test, predictions)


##########################################################################
# for use in hyperparameter tuning #
##########################################################################
def evaluate_model_numerics(model, X_test, y_test):
    """Calculates standard regression metrics and returns them as a dictionary."""
    predictions = model.predict(X_test)
    
    # Calculate metrics
    rmse = root_mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    metrics = {
        "rmse": float(rmse),
        "mae": float(mae),
        "r2_score": float(r2)
    }
    
    return metrics, predictions

def residuals_analysis_numerics(y_true, y_pred):
    """Quantifies residual behavior using statistical measures."""
    # Ensure inputs are flat arrays
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    residuals = y_true - y_pred
    
    res_mean = np.mean(residuals)
    res_std = np.std(residuals)
    
    # Skewness: > 0 means underestimating RUL, < 0 means overestimating 
    res_skew = pd.Series(residuals).skew()
    
    # Calculate % of predictions within a 10-cycle margin
    within_margin = np.mean(np.abs(residuals) <= 10) * 100
    
    res_metrics = {
        "res_mean": float(res_mean),
        "res_std": float(res_std),
        "res_skew": float(res_skew),
        "pct_within_10_cycles": float(within_margin)
    }
    
    return res_metrics


## Model Training

In [14]:
# Tracking Config
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.sklearn.autolog()

# wrapper for MLflow logging
def run_mlflow_experiment(model_trainer, X_train, y_train, X_test, y_test, run_name="Model Experiment"):
    """
    Trains, evaluates, and logs a model to MLflow.
    
    Args:
        model_trainer: A function that trains and returns a model.
        X_train, y_train: Training data.
        X_test, y_test: Testing data.
        run_name: String name for the MLflow run.
    """
    with mlflow.start_run(run_name=run_name) as run:
        # Train model
        model = model_trainer(X_train, y_train)
        mlflow.sklearn.log_model(model, "model")
        # evaluate performance
        eval_metrics, _ = evaluate_model_numerics(model, X_test, y_test)
        mlflow.log_metrics(eval_metrics)
        # Analyze residuals
        predictions = model.predict(X_test)
        res_metrics = residuals_analysis_numerics(y_test, predictions)
        mlflow.log_metrics(res_metrics)
        # closing
        run_id = run.info.run_id
        print(f"Successfully logged '{run_name}' under Run ID: {run_id}")
        return model, run_id, eval_metrics, res_metrics

# Usage:
# model, rid = run_mlflow_experiment(train_decision_tree, X_train, y_train, X_test, y_test, "Decision Tree Regressor")

In [11]:
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: sqlite:///mlflow.db


### Decision Tree Regressor

In [15]:
import os

# Hyper parameter tuning
# Experiment Setup
experiment_name = "Decision_Tree_Hyperparameter_Tuning"
artifact_uri = f"file://{os.path.join(os.getcwd(), 'mlruns_artifacts')}"

if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(name=experiment_name, artifact_location=artifact_uri)

mlflow.set_experiment(experiment_name)

    # Depth:
results = []
# Example: Testing different depths for your Decision Tree
for i in [1, 2, 3, 4, 5]:
    # Define a custom trainer target hyperparameter
    def train_dt_with_depth(X, y):
        model = DecisionTreeRegressor(max_depth=i, random_state=42)
        model.fit(X, y)
        return model

    # Run the experiment
    model, rid, eval_met, res_met = run_mlflow_experiment(
        train_dt_with_depth, 
        X_train, y_train, 
        X_val, y_val, 
        run_name=f"DT_Depth_{i}"
    )
    
    # Track results (pseudo-code)
    results.append({
        "max_depth": i,
        "run_id": rid,
        "rmse": eval_met['rmse'],
        "skew": res_met['res_skew']
    })



2026/04/23 21:56:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/23 21:56:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:56:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged 'DT_Depth_1' under Run ID: d01a6b304e904e92a10e57382994cf4c


2026/04/23 21:56:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/23 21:56:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:56:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged 'DT_Depth_2' under Run ID: f270f5a29ab5451e983bb44cda0bf1e2


2026/04/23 21:57:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/23 21:57:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:57:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged 'DT_Depth_3' under Run ID: 609370df57194e62aead3c1a53270c3c


2026/04/23 21:57:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/23 21:57:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:57:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged 'DT_Depth_4' under Run ID: 55b3c76df76f4b8da0a1c4258219eaf4


2026/04/23 21:57:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/23 21:57:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:57:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged 'DT_Depth_5' under Run ID: 7aa16a6a551245f59d07ede6079cd7db


### Random Forest Regressor

In [ ]:
def train_random_forest(X_train, y_train):
    # adjust hyperparameters as needed
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return model

# Prepare the data for Random Forest
X_train = df_unit1.drop(columns=["Unit Number", "Time, In Cycles", "RUL"])
y_train = df_unit1["RUL"]

# Start a new MLflow run for Random Forest Regressor
with mlflow.start_run(run_name="Random Forest Regressor") as rf_run:
    # Train the Random Forest model
    model_rf = train_random_forest(X_train, y_train)
    # Evaluate Random Forest model
    evaluate_model(model_rf, X_train, y_train)
    residuals_analysis(y_train, model_rf.predict(X_train))
    run_id = mlflow.last_active_run().info.run_id
    print(f"Logged data and model in run {run_id}")
    print("View results by running 'mlflow ui' and visiting http://localhost:5000")

### XGBoost Regressor

In [ ]:
def train_gradient_boosting(X_train, y_train):
    # adjust hyperparameters as needed
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    return model

# Prepare the data for Gradient Boosting
X_train = df_unit1.drop(columns=["Unit Number", "Time, In Cycles", "RUL"])
y_train = df_unit1["RUL"]

# Start a new MLflow run for Gradient Boosting Regressor
with mlflow.start_run(run_name="Gradient Boosting Regressor") as gb_run:
    # Train the Gradient Boosting model
    model_gb = train_gradient_boosting(X_train, y_train)

    # Evaluate the Gradient Boosting model
    evaluate_model(model_gb, X_train, y_train)
    residuals_analysis(y_train, model_gb.predict(X_train))
    run_id = mlflow.last_active_run().info.run_id
    print(f"Logged data and model in run {run_id}")
    print("View results by running 'mlflow ui' and visiting http://localhost:5000")